# 13.11. 全卷积网络

- 全卷积网络采用卷积神经网络实现了从图像像素到**像素类别**的变换 
- 全卷积网络将中间层特征图的高和宽变换回输入图像的尺寸：这是通过转置卷积实现的。 
- 通道维的输出即该位置对应像素的类别预测。

In [1]:
%matplotlib inline
import torch
import torchvision
from torch import nn
from torch.nn import functional as F
from d2l import torch as d2l

## 13.11.1. 构造模型

In [ ]:
pretrained_net = torchvision.models.resnet18(pretrained=True)
list(pretrained_net.children())[-3:]

- `children()`：取出模型的各个直接子模块。
- `list(...)`：把它们转换成列表。
- `[-3:]` 表示从倒数第 3 个元素开始，一直取到列表末尾。

In [ ]:
net =nn.Sequential(*list(pretrained_net.children())[: -2]) 
 # 我去，这个[: -2]又是什么意思 ，一直到倒数第2个元素之前的所有模块，把最后两个模块去掉。

In [ ]:
X = torch.rand(size=(1, 3, 320, 480))
net(X).shape

In [ ]:
num_classes=21
net.add_module('final_conv' , nn.Conv2d(512,num_classes ,kernel_size=1))
net.add_module('transpose_conv', nn.ConvTranspose2d(num_classes, num_classes,
                                    kernel_size=64, padding=16, stride=32))

## 13.11.2. 初始化转置卷积层

- 将图像放大，即上采样（upsampling）。 双线性插值（bilinear interpolation） 是常用的上采样方法之一，它也经常用于初始化转置卷积层。
- 

In [ ]:
def bilinear_kernel(in_channels, out_channels, kernel_size):
    factor = (kernel_size + 1) // 2
    if kernel_size % 2 == 1:
        center = factor - 1
    else:
        center = factor - 0.5
    og = (torch.arange(kernel_size).reshape(-1, 1),
          torch.arange(kernel_size).reshape(1, -1))
    filt = (1 - torch.abs(og[0] - center) / factor) * \
           (1 - torch.abs(og[1] - center) / factor)
    weight = torch.zeros((in_channels, out_channels,
                          kernel_size, kernel_size))
    weight[range(in_channels), range(out_channels), :, :] = filt
    return weight

-  我们构造一个将输入的高和宽放大2倍的转置卷积层，并将其卷积核用bilinear_kernel函数初始化。

In [ ]:
conv_trans = nn.ConvTranspose2d(3, 3, kernel_size=4, padding=1, stride=2,
                                bias=False)
conv_trans.weight.data.copy_(bilinear_kernel(3, 3, 4))

In [ ]:
# 读取图像X，将上采样的结果记作Y
img = torchvision.transforms.ToTensor()(d2l.Image.open('./img/catdog.jpg'))
X = img.unsqueeze(0)
Y = conv_trans(X)
out_img = Y[0].permute(1, 2, 0).detach()

d2l.set_figsize()
print('input image shape:', img.permute(1, 2, 0).shape)
d2l.plt.imshow(img.permute(1, 2, 0));
print('output image shape:', out_img.shape)
d2l.plt.imshow(out_img);

In [ ]:
W = bilinear_kernel(num_classes, num_classes, 64)
net.transpose_conv.weight.data.copy_(W);

## 13.11.3. 读取数据集

In [ ]:
batch_size, crop_size = 32, (320, 480)
train_iter, test_iter = d2l.load_data_voc(batch_size, crop_size)

## 13.11.4. 训练

- 因为我们使用转置卷积层的通道来预测像素的类别，所以需要在损失计算中指定通道维。
-  此外，模型基于每个像素的预测类别是否正确来计算准确率。

In [ ]:
def loss(inputs, targets):
    return F.cross_entropy(inputs, targets, reduction='none').mean(1).mean(1)

num_epochs, lr, wd, devices = 5, 0.001, 1e-3, d2l.try_all_gpus()
trainer = torch.optim.SGD(net.parameters(), lr=lr, weight_decay=wd)
d2l.train_ch13(net, train_iter, test_iter, loss, trainer, num_epochs, devices)

## 13.11.5. 预测

- 在预测时，我们需要将输入图像在各个通道做标准化，并转成卷积神经网络所需要的四维输入格式。

In [ ]:
def predict(img):
    X = test_iter.dataset.normalize_image(img).unsqueeze(0)
    pred = net(X.to(devices[0])).argmax(dim=1) # 这是什么 ,这个pred是什么意思
    return pred.reshape(pred.shape[1], pred.shape[2]) 

- `argmax` 的作用是：返回最大值所在的位置（索引）。
- `dim=1` 表示沿着第`1` 维寻找最大值。这里第 `1` 维正好是类别通道。

In [ ]:
def label2image(pred):
    colormap =torch.tensor(d2l.VOC_COLORMAP,device = devices[0]) # 这个device[0]是什么意思
    X =pred.long() # 呃，着什么语法
    return colormap[X,:]

- `.long()` 是 PyTorch 的数据类型转换方法，把张量转换为 `torch.int64`整数类型。
- 


In [ ]:
voc_dir = d2l.download_extract('voc2012', 'VOCdevkit/VOC2012')
test_images, test_labels = d2l.read_voc_images(voc_dir, False)
n, imgs = 4, []
for i in range(n):
    crop_rect = (0, 0, 320, 480)
    X = torchvision.transforms.functional.crop(test_images[i], *crop_rect)
    pred = label2image(predict(X))
    imgs += [X.permute(1,2,0), pred.cpu(),
             torchvision.transforms.functional.crop(
                 test_labels[i], *crop_rect).permute(1,2,0)]
d2l.show_images(imgs[::3] + imgs[1::3] + imgs[2::3], 3, n, scale=2);